# Task 1 — Predict Restaurant Ratings
**Internship:** Cognifyz Technologies — Machine Learning  
**Objective:** Build a regression model to predict the aggregate rating of a restaurant.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/vp763/ML_Project_cognifyz/refs/heads/main/Dataset%20.csv'
data = pd.read_csv(url)
print(f"Dataset shape: {data.shape}")
data.head()

## 3. Preprocessing

In [ ]:
# Check missing values
data.isnull().sum()

In [ ]:
# Fill missing cuisines
data['Cuisines'] = data['Cuisines'].fillna('Unknown')

# Encode binary columns
binary_cols = ['Has Table booking', 'Has Online delivery', 'Is delivering now', 'Switch to order menu']
for col in binary_cols:
    data[col] = data[col].map({'Yes': 1, 'No': 0})

# Label encode categorical columns
cat_cols = ['City', 'Cuisines', 'Currency', 'Rating color', 'Rating text']
le = LabelEncoder()
for col in cat_cols:
    data[col] = le.fit_transform(data[col].astype(str))

print("Preprocessing complete.")

## 4. Feature Selection & Train-Test Split

In [ ]:
# Drop columns not useful for prediction
drop_cols = ['Restaurant ID', 'Restaurant Name', 'Country Code',
             'Address', 'Locality', 'Locality Verbose', 'Rating color', 'Rating text']
data.drop(columns=drop_cols, inplace=True)

X = data.drop('Aggregate rating', axis=1)
y = data['Aggregate rating']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

## 5. Model Training & Evaluation

In [ ]:
models = {
    'Linear Regression'  : LinearRegression(),
    'Decision Tree'      : DecisionTreeRegressor(random_state=42)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    r2  = r2_score(y_test, y_pred)
    results[name] = {'MSE': round(mse, 4), 'R2': round(r2, 4)}
    print(f"{name:22s} | MSE: {mse:.4f} | R2: {r2:.4f}")

In [ ]:
# Check for overfitting in Decision Tree
train_pred = models['Decision Tree'].predict(X_train)
print(f"Decision Tree Train R2 : {r2_score(y_train, train_pred):.4f}")
print(f"Decision Tree Test  R2 : {results['Decision Tree']['R2']}")

## 6. Visualizations

In [ ]:
# Model comparison — MSE and R2
results_df = pd.DataFrame(results).T

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

results_df['R2'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('R² Score')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R²')

results_df['MSE'].plot(kind='bar', ax=axes[1], color='tomato')
axes[1].set_title('Mean Squared Error')
axes[1].set_ylabel('MSE')

plt.suptitle('Model Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
# Feature importance — Decision Tree
feat_imp = pd.Series(
    models['Decision Tree'].feature_importances_,
    index=X.columns
).sort_values(ascending=False)

plt.figure(figsize=(10, 4))
feat_imp.plot(kind='bar', color='mediumseagreen')
plt.title('Feature Importance — Decision Tree')
plt.ylabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("\nTop 3 features:")
print(feat_imp.head(3))

In [ ]:
# Actual vs Predicted — Decision Tree
y_pred_dt = models['Decision Tree'].predict(X_test)

plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_dt, alpha=0.3, color='royalblue', s=15)
plt.plot([0, 5], [0, 5], 'r--', linewidth=1)
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title('Actual vs Predicted — Decision Tree')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150)
plt.show()

## 7. Results & Interpretation

| Model | MSE | R² |
|---|---|---|
| Linear Regression | 1.6403 | 0.2794 |
| Decision Tree | 0.1775 | 0.9220 |

**Best model:** Decision Tree Regressor

**Overfitting check:** Train R² = 1.0 vs Test R² = 0.92 — slight overfitting, but test performance is acceptable.

**Most influential features:**
1. `Votes` — 94.67% (dominant feature)
2. `Longitude` — 1.92%
3. `Latitude` — 1.19%

**Key insight:** Number of votes is the strongest predictor of restaurant ratings. Restaurants with higher engagement tend to have higher aggregate ratings, suggesting a strong link between popularity and perceived quality.